# Few-Shot Example Selection

## Overview

Few-shot prompting requires representative examples that demonstrate how flaky and non-flaky tests should be analyzed.

Rather than selecting examples arbitrarily, representative examples were selected from the preprocessed dataset using quantitative criteria. The goal was to choose examples that are representative of typical test cases while avoiding unusually small or extremely large artifacts that could bias the evaluation.

The selected examples will be used only as demonstrations in the few-shot prompts. These examples are removed from the evaluation dataset to prevent data leakage during model evaluation.

In [58]:
from pathlib import Path

import pandas as pd

In [59]:
# ============================================================
# Load Preprocessed Dataset
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "context_augmented_dataset_preprocessed.jsonl"
)

working_df = pd.read_json(
    DATASET_PATH,
    lines=True
)

working_df["isFlaky"] = working_df["isFlaky"].astype(bool)

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)
print(f"Total Records : {len(working_df)}")

DATASET LOADED
Total Records : 2743


## Compute Artifact Statistics

The dataset contains multiple contextual artifacts with varying sizes.

To identify representative examples, descriptive statistics are first computed for each artifact. These statistics are later used to identify examples whose artifact sizes are close to the central tendency of the dataset.

In [61]:
candidate_df = working_df.copy()

candidate_df["test_length"] = candidate_df["test_code"].str.len()

candidate_df["helper_count"] = candidate_df["helper_methods_json"].apply(
    lambda x: len(x) if isinstance(x, dict) else 0
)

candidate_df["failure_length"] = candidate_df["failure_log"].str.len()

candidate_df["production_classes"] = candidate_df["code_under_test_json"].apply(
    lambda x: len(x) if isinstance(x, dict) else 0
)

candidate_df["artifact_completeness"] = (
    (candidate_df["test_length"] > 0).astype(int)
    + (candidate_df["failure_length"] > 0).astype(int)
    + (candidate_df["production_classes"] > 0).astype(int)
    + (candidate_df["helper_count"] > 0).astype(int)
)

In [20]:
summary = candidate_df[
    [
        "test_length",
        "helper_count",
        "failure_length",
        "production_classes",
        "artifact_completeness"
    ]
].describe()

display(summary)

,test_length,helper_count,failure_length,production_classes,artifact_completeness
count,2745.000000,2745.000000,2745.000000,2745.000000,2745.000000
mean,710.868488,0.856466,3226.079781,10.938069,3.491075
std,913.269735,1.310065,6210.939116,23.031142,0.520019
min,55.000000,0.000000,0.000000,0.000000,2.000000
25%,296.000000,0.000000,1625.000000,1.000000,3.000000
50%,458.000000,1.000000,2933.000000,4.000000,4.000000
75%,851.000000,1.000000,3395.000000,9.000000,4.000000
max,21275.000000,20.000000,165904.000000,312.000000,4.000000


## Identify Representative Candidate Examples

Representative candidates are selected using the following objective criteria:

- Complete contextual information (artifact completeness = 4).
- Artifact sizes close to the median of the dataset.
- Moderate production code context.
- Representative of the corresponding class.

These criteria reduce the likelihood of selecting trivial or unusually large examples while ensuring that the demonstrations resemble a typical sample from the dataset.

In [21]:
# ============================================================
# Dataset Medians
# ============================================================

median_test = candidate_df["test_length"].median()
median_failure = candidate_df["failure_length"].median()
median_production = candidate_df["production_classes"].median()

# ============================================================
# Normalized Distance from Dataset Median
# ============================================================

candidate_df["distance_from_median"] = (

    abs(candidate_df["test_length"] - median_test) / median_test

    + abs(candidate_df["failure_length"] - median_failure) / median_failure

    + abs(candidate_df["production_classes"] - median_production) / median_production

)

## Construct Candidate Pools

Candidate pools are created by separating flaky and non-flaky test cases.

The representative ranking is then performed independently within each class to identify suitable demonstration examples.

In [35]:
# ============================================================
# Construct Candidate Pools
# ============================================================

flaky_candidates = candidate_df[
    candidate_df["isFlaky"]
].copy()

non_flaky_candidates = candidate_df[
    ~candidate_df["isFlaky"]
].copy()

print(f"Flaky Candidates     : {len(flaky_candidates)}")
print(f"Non-Flaky Candidates : {len(non_flaky_candidates)}")

Flaky Candidates     : 1107
Non-Flaky Candidates : 1638


In [37]:
flaky_ranked = (
    flaky_candidates
    .sort_values("distance_from_median")
    .copy()
)

non_flaky_ranked = (
    non_flaky_candidates
    .sort_values("distance_from_median")
    .copy()
)

## Top Representative Candidates

The highest ranked flaky and non-flaky candidates are inspected manually before final selection.

Although the ranking identifies statistically representative records, a manual inspection is still required to ensure that the selected examples are understandable, contain meaningful contextual artifacts, and clearly illustrate the intended reasoning process.

In [38]:
columns = [
    "test_id",
    "issue_category",
    "test_length",
    "helper_count",
    "failure_length",
    "production_classes",
    "distance_from_median"
]

display(flaky_ranked[columns].head(15))

,test_id,issue_category,test_length,helper_count,failure_length,production_classes,distance_from_median
180,fastjsonf164d85test_for_issue4,Implementation Dependent,505,0,2896,4,0.115235
758,OpenRefinemaina68ba3btestSelectedEmptyChoice,Implementation Dependent,541,1,2962,4,0.191110
757,OpenRefinemaina68ba3bserializeListFacet,Implementation Dependent,547,1,2957,4,0.202506
789,jolokiasupportspring275165bsystemProperties,Implementation Dependent,535,1,3088,4,0.220969
182,fastjson803fd73test_for_issue8,Implementation Dependent,318,0,2685,4,0.390232
687,snakeyaml626340btestBoolOutAsEmpty32,Implementation Dependent,407,2,2710,3,0.437385
904,wildflynaming3a83b7b10,Order Dependent,363,2,2925,3,0.460151
67,avrolangjavaavro7fd098atestRecord,Implementation Dependent,286,1,3459,4,0.554884
48,bladebladecoreaa32ce9testCreateRouteBuilder,Implementation Dependent,517,0,3525,3,0.580662
670,slingorgapacheslingservletsget1cea3aetestBoole...,Implementation Dependent,516,1,3662,5,0.625189


In [39]:
columns = [
    "test_id",
    "test_length",
    "helper_count",
    "failure_length",
    "production_classes",
    "distance_from_median"
]

display(non_flaky_ranked[columns].head(15))

,test_id,test_length,helper_count,failure_length,production_classes,distance_from_median
1951,JacksonCore-21-2,452,4,2786,4,0.063220
1959,JacksonCore-22-7,490,4,2888,4,0.085212
1956,JacksonCore-22-4,494,4,2889,4,0.093604
1125,Cli-15-1,447,1,2704,4,0.102095
1111,Cli-4-2,446,0,2707,4,0.103255
1361,Closure-64,414,1,2881,4,0.113799
1155,Cli-31,485,0,2729,4,0.128505
2157,Jsoup-25,444,0,3235,4,0.133534
2218,Jsoup-61-2,458,0,3347,4,0.141152
2695,Time-12-2,400,0,2875,4,0.146413


## Candidate Inspection

The ranking procedure identifies representative candidates based on artifact sizes.

However, multiple highly ranked candidates may originate from the same software project and therefore represent nearly identical testing scenarios.

To improve diversity, a small number of highly ranked candidates from different projects are manually inspected before selecting the final demonstrations.

In [40]:
def inspect_candidate(index):

    sample = working_df.loc[index]

    print("=" * 80)
    print(f"Index : {index}")
    print(f"Test ID : {sample['test_id']}")
    print(f"Flaky : {sample['isFlaky']}")
    print(f"Category : {sample['issue_category']}")
    print("=" * 80)

    print("\nArtifact Summary")
    print("-" * 80)
    print(f"Test Length         : {len(sample['test_code'])}")
    print(f"Helper Methods      : {len(sample['helper_methods_json']) if isinstance(sample['helper_methods_json'], dict) else 0}")
    print(f"Failure Log Length  : {len(sample['failure_log'])}")
    print(f"Production Classes  : {len(sample['code_under_test_json']) if isinstance(sample['code_under_test_json'], dict) else 0}")

    print("\n" + "=" * 80)
    print("TEST CODE")
    print("=" * 80)
    print(sample["test_code"])

    print("\n" + "=" * 80)
    print("HELPER METHODS")
    print("=" * 80)
    print(sample["helper_methods_json"])

    print("\n" + "=" * 80)
    print("FAILURE LOG")
    print("=" * 80)
    print(sample["failure_log"])

    print("\n" + "=" * 80)
    print("PRODUCTION CODE")
    print("=" * 80)
    print(sample["code_under_test_json"])

In [ ]:
inspect_candidate(758)
inspect_candidate(789)
inspect_candidate(904)

Index : 758
Test ID : OpenRefinemaina68ba3btestSelectedEmptyChoice
Flaky : True
Category : Implementation Dependent

Artifact Summary
--------------------------------------------------------------------------------
Test Length         : 541
Helper Methods      : 1
Failure Log Length  : 2962
Production Classes  : 4

TEST CODE
@Test
    public void testSelectedEmptyChoice() throws IOException {
        Project project = createCSVProject("Column A\n" +
                "a\n" +
                "c\n" +
                "e");
        Engine engine = new Engine(project);

        ListFacetConfig facetConfig = ParsingUtilities.mapper.readValue(jsonConfig, ListFacetConfig.class);
        Facet facet = facetConfig.apply(project);
        facet.computeChoices(project, engine.getAllFilteredRows());
        TestUtils.isSerializedTo(facet, selectedEmptyChoiceFacet);
    }

HELPER METHODS
{'createCSVProject': 'protected Project createCSVProject(String input) {\n        return createCSVProject("test pro

In [ ]:
inspect_candidate(1951)
inspect_candidate(1125)
inspect_candidate(1361)

Index : 1951
Test ID : JacksonCore-21-2
Flaky : False
Category : Non-Flaky

Artifact Summary
--------------------------------------------------------------------------------
Test Length         : 452
Helper Methods      : 4
Failure Log Length  : 2786
Production Classes  : 4

TEST CODE
@SuppressWarnings("resource")
    public void testBasicSingleMatchFilteringWithPath() throws Exception
    {
        JsonParser p0 = JSON_F.createParser(SIMPLE);
        JsonParser p = new FilteringParserDelegate(p0,
               new NameMatchFilter("value"),
                   true,
                   false
                );

        String result = readAndWrite(JSON_F, p);
        assertEquals(aposToQuotes("{'ob':{'value':3}}"), result);
    }

HELPER METHODS
{'NameMatchFilter': 'public NameMatchFilter(String... names) {\n            _names = new HashSet<String>(Arrays.asList(names));\n        }', 'aposToQuotes': 'protected String aposToQuotes(String json) {\n        return json.replace("\'", "\\"");

## Final Example Selection

The highest ranked candidates were manually inspected to identify representative demonstrations for few-shot prompting.

The final examples were selected based on the following considerations:

- close to the median artifact sizes of the dataset,
- complete contextual information (test code, helper methods, failure log, and production code),
- concise and readable source code,
- informative failure logs,
- manageable production code size,
- deterministic behaviour that clearly distinguishes flaky and non-flaky tests,
- suitability as teaching examples for Large Language Models.

Following this inspection, one flaky example and one non-flaky example were selected and fixed for all subsequent few-shot experiments.

Using the same demonstrations across all prompt configurations ensures that any performance differences observed during evaluation are attributable to the prompting strategy and contextual information rather than differences in the demonstration examples themselves.

In [41]:
FLAKY_EXAMPLE_INDEX = 758
NON_FLAKY_EXAMPLE_INDEX = 1951

selected_examples = working_df.loc[
    [FLAKY_EXAMPLE_INDEX, NON_FLAKY_EXAMPLE_INDEX]
]

selected_examples[
    [
        "test_id",
        "isFlaky",
        "issue_category"
    ]
]

,test_id,isFlaky,issue_category
758,OpenRefinemaina68ba3btestSelectedEmptyChoice,True,Implementation Dependent
1951,JacksonCore-21-2,False,Non-Flaky


## Create Evaluation Dataset

The selected demonstration examples are removed from the evaluation dataset to eliminate data leakage.

The resulting evaluation dataset is subsequently used throughout all LLM experiments, while the selected examples are used exclusively as fixed demonstrations for few-shot prompting.

In [42]:
evaluation_df = working_df.drop(
    index=[
        FLAKY_EXAMPLE_INDEX,
        NON_FLAKY_EXAMPLE_INDEX
    ]
)

print(f"Original Dataset   : {len(working_df)}")
print(f"Evaluation Dataset : {len(evaluation_df)}")

Original Dataset   : 2745
Evaluation Dataset : 2743


In [43]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "context_augmented_dataset_evaluation.jsonl"
)

evaluation_df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True
)

print(f"Saved evaluation dataset to:\n{OUTPUT_PATH}")

Saved evaluation dataset to:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\datasets\context_augmented_dataset_evaluation.jsonl


# Second Selection Round

## Motivation

The initially selected representative examples were validated using an expert language model in the subsequent notebook.

During validation, the initial non-flaky example was successfully verified. However, the initial flaky example was incorrectly classified by the expert model when only the test code was provided.

Since few-shot demonstrations must accurately represent the dataset labels, the initial flaky example was rejected.

A second representative flaky example is therefore selected using the same ranking procedure while keeping the validated non-flaky example unchanged.

In [44]:
SECOND_FLAKY_INDEX = 789

inspect_candidate(SECOND_FLAKY_INDEX)

Index : 789
Test ID : jolokiasupportspring275165bsystemProperties
Flaky : True
Category : Implementation Dependent

Artifact Summary
--------------------------------------------------------------------------------
Test Length         : 535
Helper Methods      : 1
Failure Log Length  : 3088
Production Classes  : 4

TEST CODE
@Test
    public void systemProperties() throws Exception {
        checkSystemPropertyMode("fallback","/jol/","/j4p/","/j4p/");
        checkSystemPropertyMode("fallback","/jol/",null,"/jol/");
        checkSystemPropertyMode("fallback",null,null,"/jolokia/");

        checkSystemPropertyMode("override","/jol/","/j4p/","/jol/");
        checkSystemPropertyMode("override","/jol/",null,"/jol/");
        checkSystemPropertyMode("override",null,null,"/jolokia/");

        checkSystemPropertyMode(null,"/jol/",null,"/jolokia/");
    }

HELPER METHODS
{'checkSystemPropertyMode': 'private void checkSystemPropertyMode(String mode,String propContext,String configContext,Stri

In [45]:
selected_examples = working_df.loc[
    [
        FLAKY_EXAMPLE_INDEX,
        NON_FLAKY_EXAMPLE_INDEX
    ]
]

selected_examples[
    [
        "test_id",
        "isFlaky"
    ]
]

,test_id,isFlaky
758,OpenRefinemaina68ba3btestSelectedEmptyChoice,True
1951,JacksonCore-21-2,False


## Create Updated Evaluation Dataset

To prevent information leakage during model evaluation, the final selected demonstration examples are removed from the evaluation dataset.

The resulting dataset will be used during the feasibility analysis and subsequent LLM evaluation.

In [46]:
evaluation_df = working_df.drop(
    index=[
        FLAKY_EXAMPLE_INDEX,
        NON_FLAKY_EXAMPLE_INDEX
    ]
)

print(f"Evaluation Records : {len(evaluation_df)}")

Evaluation Records : 2743


In [48]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "context_augmented_dataset_evaluation.jsonl"
)

evaluation_df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True
)

print("✓ Final evaluation dataset saved.")
print(OUTPUT_PATH)

✓ Final evaluation dataset saved.
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\datasets\context_augmented_dataset_evaluation.jsonl


## Revision of Demonstration Selection Strategy

The first two representative flaky candidates were rejected during expert validation because the expert model could not reliably infer flaky behaviour from the test code alone.

Although these examples were representative in terms of dataset statistics, they did not provide sufficiently observable flaky characteristics for code-only prompting.

Therefore, the demonstration selection strategy was refined.

For the code-only prompt configuration, replacement candidates are selected based on the presence of explicit flaky indicators within the test code (e.g., concurrency or timing constructs), while maintaining the previously validated non-flaky demonstration.

## Select Code-Oriented Replacement Candidate

Following the refinement of the demonstration selection strategy, a new flaky example is selected specifically for the code-only prompt configuration.

Unlike the previous representative examples, the replacement candidate is chosen because the flaky behaviour is explicitly observable within the test code (e.g., concurrency or timing constructs). Such characteristics enable the expert model to infer the correct classification without requiring additional contextual artifacts.

The previously validated non-flaky demonstration remains unchanged.

In [52]:
FINAL_FLAKY_TEST_ID = "CURATOR-671"

FINAL_NON_FLAKY_TEST_ID = "JacksonCore-21-2"

In [50]:
replacement_example = working_df[
    working_df["test_id"] == REPLACEMENT_TEST_ID
].iloc[0]

display(
    replacement_example[
        [
            "test_id",
            "issue_category",
            "isFlaky"
        ]
    ]
)

test_id              CURATOR-671
issue_category    Time Dependent
isFlaky                     True
Name: 1082, dtype: object

In [53]:
final_examples = working_df[

    working_df["test_id"].isin(

        [
            FINAL_FLAKY_TEST_ID,
            FINAL_NON_FLAKY_TEST_ID
        ]

    )

]

display(

    final_examples[
        [
            "test_id",
            "issue_category",
            "isFlaky"
        ]
    ]

)

,test_id,issue_category,isFlaky
1082,CURATOR-671,Time Dependent,True
1951,JacksonCore-21-2,Non-Flaky,False


## Create Final Evaluation Dataset

The final selected demonstration examples are removed from the evaluation dataset to prevent data leakage during LLM evaluation.

This dataset will be used throughout the subsequent experiments.

In [54]:
final_dataset = working_df[

    ~working_df["test_id"].isin(

        [
            FINAL_FLAKY_TEST_ID,
            FINAL_NON_FLAKY_TEST_ID
        ]

    )

].copy()

print(len(final_dataset))

2743


In [63]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "context_augmented_dataset_preprocessed.jsonl"
)

final_dataset.to_json(

    OUTPUT_PATH,
    orient="records",
    lines=True

)

print("✓ Final evaluation dataset saved.")

✓ Final evaluation dataset saved.
